### Configure API Client

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("BINANCE_API_KEY")
api_secret = os.getenv("BINANCE_SECRET_KEY")

In [ ]:
from binance.client import Client
client = Client(api_key, api_secret, testnet=True)

### Load Data

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("../data/raw_ohlcv/BTCUSDT-1h-2017-08-17.csv", index_col='Date', parse_dates=['Date'])
df = df[["Close", "Volume"]].copy()

In [ ]:
df.head()

### Calculating Returns

In [ ]:
import numpy as np

In [ ]:
df["Return"] = df["Close"].div(df["Close"].shift(1))     # This is the return factor from that period's interval
df["Return"] = np.log(df["Return"])    # Converts to log returns for additivity and scale
df

### Visualising Data


In [ ]:
import matplotlib.pyplot as plt

In [ ]:
df.Close.loc['2021':'2023'].plot(figsize=(12,8))
plt.show()

In [ ]:
fig, axs = plt.subplots(2,2, figsize=(12,8))

df.Close.plot(ax=axs[0,0])
df.Volume.plot(ax=axs[1,0])
df.Close.loc['2021':'2023'].plot(ax=axs[1,1])
df.Return.plot(ax=axs[0,1])

plt.tight_layout()
plt.show()

### Buy and Hold Strategy

In [ ]:
# Normalise Close price with respect to the first price - effectively the multiple at that time
normalised_price_series = df.Close / df.Close.iloc[0]     

In [ ]:
# Calculating the multiple directly
multiple = np.exp(df["Return"].sum())
multiple

In [ ]:
# Cumulative return from buying and holding
df['C_return'] = df['Return'].cumsum().apply(np.exp)
df

### Backtesting Random Strategy

#### Backtester

In [ ]:
# Using a subset of data (only 2021) for testing
test_data = df.loc['2021'].copy()
test_data['C_return'] = test_data['Return'].cumsum().apply(np.exp)
test_data

In [ ]:
test_data['Position'] = 1

In [ ]:
# Random strategy - sells when the close price is mod 10
sell_condition = test_data['Close'] % 10 == 0

In [ ]:
test_data.loc[sell_condition, 'Position'] = 0

In [ ]:
# Checks to see how many times the strategy is not holding
test_data.Position.value_counts()

In [ ]:
test_data['Strategy'] = test_data['Position'].shift(1) * test_data['Return']

In [ ]:
test_data['C_strategy'] = test_data['Strategy'].cumsum().apply(np.exp)

In [ ]:
test_data[['C_return', 'C_strategy']].plot(figsize=(12,8))
plt.legend(['Buy and Hold', 'Your Strategy'])
plt.show()

#### Performance Metrics

In [ ]:
# Number of trading periods
trading_periods = 8760

##### Annaulised Mean

In [ ]:
ann_mean_log = test_data[['Return', 'Strategy']].mean() * trading_periods
ann_mean = np.exp(ann_mean_log) - 1
ann_mean

##### Standard Deviation

In [ ]:
ann_std = test_data[['Return', 'Strategy']].std() * np.sqrt(trading_periods)
ann_std

##### Sharpe Ratio

In [ ]:
sharpe = (ann_mean / ann_std)
sharpe